In [2]:
import pandas as pd
import time
import os

In [3]:
def run_it_control_monitor(filepath):
    """
    Main function to analyze IT control effectiveness.
    Identifies 'Out-of-Sequence' changes and 'Segregation of Duties' violations.
    """
    print("="*60)
    print("STARTING IT CONTROL EFFECTIVENESS MONITOR")
    print("="*60)
    
    start_time = time.time()

    # 1. DATA INGESTION
    if not os.path.exists(filepath):
        print(f"CRITICAL ERROR: File '{filepath}' not found. Check the path.")
        return

    try:
        # Loading the dataset (Expected columns: case_id, activity, user_id, timestamp)
        df = pd.read_csv(filepath)
        print(f"LOADED: {len(df):,} total log records.")
    except Exception as e:
        print(f"INGESTION ERROR: {e}")
        return

    # 2. PRE-PROCESSING & SEQUENCING
    # Ensure timestamps are in datetime format and data is sorted by case and time
    df['timestamp'] = pd.to_datetime(
        df['timestamp'],
        format='mixed',
        utc=True,
        errors='coerce'
    )

    df = df.sort_values(by=['case_id', 'timestamp'])
    
    # Reconstructing process lifecycles
    cases = df.groupby('case_id')
    total_cases = len(cases)
    
    cm_failures = 0  # Change Management Failure Counter
    ac_failures = 0  # Access Control (SoD) Failure Counter
    detailed_results = []

    print(f"ANALYSING: {total_cases:,} unique business process cases...")

    # 3. EVALUATION ENGINE (The Logic Layer)
    for case_id, group in cases:
        activities = group['activity'].tolist()
        
        # --- Control A: Change Management (Sequence Integrity) ---
        # Rule: 'Activation' must always be preceded by an 'Approval' activity
        cm_status = "PASS"
        if 'Application Activated' in activities:
            if 'Application Approved' not in activities:
                cm_status = "FAIL: Missing Approval"
                cm_failures += 1
            elif activities.index('Application Approved') > activities.index('Application Activated'):
                cm_status = "FAIL: Out-of-Sequence"
                cm_failures += 1
        
        # --- Control B: Access Control (Segregation of Duties - SoD) ---
        # Rule: The user who 'Submits' cannot be the same user who 'Approves'
        ac_status = "PASS"
        sub_users = set(group[group['activity'] == 'Application Submitted']['user_id'])
        app_users = set(group[group['activity'] == 'Application Approved']['user_id'])
        
        # Intersection check for common user ID in both activities
        if sub_users.intersection(app_users):
            ac_status = "FAIL: SoD Violation"
            ac_failures += 1
            
        detailed_results.append({
            'case_id': case_id,
            'Change_Management': cm_status,
            'Access_Control': ac_status
        })

    # 4. FINAL RESULTS & METRICS
    end_time = time.time()
    cm_rate = ((total_cases - cm_failures) / total_cases) * 100
    ac_rate = ((total_cases - ac_failures) / total_cases) * 100
    
    print("\n" + "="*50)
    print("                FINAL AUDIT RESULTS")
    print("="*50)
    print(f"Total Cases Analyzed:   {total_cases:,}")
    print(f"Processing Time:        {end_time - start_time:.2f} seconds")
    print("-" * 50)
    print(f"Change Mgmt Effectiveness: {cm_rate:.2f}%")
    print(f"Access Control Effectiveness: {ac_rate:.2f}%")
    print("-" * 50)

    # Threshold Check 
    for name, rate in [("Change Management", cm_rate), ("Access Control", ac_rate)]:
        status = "EFFECTIVE (GREEN)" if rate >= 98.0 else "WEAKNESS (RED)"
        print(f"Status for {name}: {status}")
    print("="*50)

    # Return results as a DataFrame for logging/evidence
    return pd.DataFrame(detailed_results)

# --- EXECUTION BLOCK ---
if __name__ == "__main__":
    FILE_NAME = 'event_log.csv' 
    results_df = run_it_control_monitor(FILE_NAME)

STARTING IT CONTROL EFFECTIVENESS MONITOR
LOADED: 262,200 total log records.
ANALYSING: 13,087 unique business process cases...

                FINAL AUDIT RESULTS
Total Cases Analyzed:   13,087
Processing Time:        28.12 seconds
--------------------------------------------------
Change Mgmt Effectiveness: 100.00%
Access Control Effectiveness: 0.00%
--------------------------------------------------
Status for Change Management: EFFECTIVE (GREEN)
Status for Access Control: WEAKNESS (RED)


In [4]:

print(os.getcwd())

C:\Users\okach
